# Figure 1: neural OT on the bimodal Gaussian target

This notebook trains `NeuralOptimalTransportPredictor` on the synthetic bimodal Gaussian dataset used by the figure-1 experiments, then plots the neural pushforward and pullback sets.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch

from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.stats import chi2

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent.parent

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from configs.datasets.synthetic.bimodal_gaussian import (
    BimodalGaussianDatasetConfig,
)
from configs.predictors.transport.neural_optimal_transport import (
    NeuralOptimalTransportPredictorConfig,
)
from configs.trainers.transport.neural_optimal_transport import (
    NeuralOptimalTransportTrainerConfig,
)
from data.datasets.synthetic.bimodal_gaussian import BimodalGaussianDataset
from data.loaders import make_xy_dataloader
from predictors.transport.neural_optimal_transport import (
    NeuralOptimalTransportPredictor,
)
from trainers.transport.neural_optimal_transport import NeuralQuantileTrainer

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
seed = 7
torch.manual_seed(seed)

device = "cpu"
dtype = "float32"

n_train = 4_096
n_calibration = 1_024
n_test = 1_024
batch_size = 256

hidden_dim = 128
num_hidden_layers = 3
epochs = 25
warmup_iterations = 1
c_transform_max_iter = 50


In [ ]:
dataset_config = BimodalGaussianDatasetConfig(
    n_train=n_train,
    n_calibration=n_calibration,
    n_test=n_test,
    x_dim=1,
    y_dim=2,
    mode_offset=1.0,
    noise_scale=1.0,
    seed=seed,
    device=device,
    dtype=dtype,
)

dataset = BimodalGaussianDataset(dataset_config)
splits = dataset.get_splits()

train_dataloader = make_xy_dataloader(
    data=splits.train,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

print("train x/y:", splits.train.x.shape, splits.train.y.shape)
print("calibration x/y:", splits.calibration.x.shape, splits.calibration.y.shape)
print("test x/y:", splits.test.x.shape, splits.test.y.shape)
print("target modes:", dataset.modes)


In [ ]:
predictor_config = NeuralOptimalTransportPredictorConfig(
    x_dim=dataset.x_dim,
    y_dim=dataset.y_dim,
    hidden_dim=hidden_dim,
    num_hidden_layers=num_hidden_layers,
    potential_type="u",
    c_transform_lr=0.25,
    c_transform_max_iter=c_transform_max_iter,
    seed=seed,
    device=device,
    dtype=dtype,
)

trainer_config = NeuralOptimalTransportTrainerConfig(
    epochs=epochs,
    learning_rate=1e-3,
    weight_decay=1e-4,
    warmup_iterations=warmup_iterations,
    grad_clip_norm=1.0,
    use_cosine_scheduler=True,
    verbose=True,
)

predictor = NeuralOptimalTransportPredictor(predictor_config)
trainer = NeuralQuantileTrainer(trainer_config)


In [ ]:
predictor = trainer.fit(
    predictor=predictor,
    dataloader=train_dataloader,
)

trainer.training_history[-5:]

In [ ]:
predictor.eval()

n_eval = 2_048
x_eval = dataset.sample_x(n_eval)
u_eval = dataset.sample_source(n_eval)
y_generated = predictor.pushforward(x=x_eval, u=u_eval)

z_test = predictor.multivariate_score(
    x=splits.test.x[:n_eval],
    y=splits.test.y[:n_eval],
)

summary = {
    "generated_mean": y_generated.mean(dim=0).detach().cpu(),
    "generated_std": y_generated.std(dim=0).detach().cpu(),
    "target_mean": splits.test.y[:n_eval].mean(dim=0).detach().cpu(),
    "target_std": splits.test.y[:n_eval].std(dim=0).detach().cpu(),
    "test_score_mean": z_test.mean(dim=0).detach().cpu(),
    "test_score_std": z_test.std(dim=0).detach().cpu(),
}

summary


In [ ]:
predictor.eval()

# The plotted sets use the conventional 1 - alpha coverage level,
# matching figure_1_discrete_ot.
alpha = 0.75
coverage_mass = 1.0 - alpha

n_radius_mc = 1_000_000
n_boundary_points = 720
plot_batch_size = 256

source_bounds = (-1.4, 1.4, -1.4, 1.4)
target_bounds = (-1.9, 1.9, -1.4, 1.4)


def calibrate_hdr_radius(mass, n_mc):
    y_mc = dataset.sample_target(n_mc)
    distances = torch.linalg.norm(
        y_mc[:, None, :] - dataset.modes[None, :, :],
        dim=-1,
    ).min(dim=1).values

    rank = int(np.ceil(mass * distances.numel()))
    rank = max(1, min(rank, distances.numel()))
    radius = torch.kthvalue(distances, rank).values
    empirical_mass = (distances <= radius).to(torch.float32).mean()
    return float(radius.detach().cpu()), float(empirical_mass.detach().cpu())


def neural_pushforward(u, batch_size=plot_batch_size):
    mapped = []
    for start in range(0, u.shape[0], batch_size):
        u_batch = u[start:start + batch_size]
        x_batch = dataset.sample_x(u_batch.shape[0])
        mapped.append(predictor.pushforward(x=x_batch, u=u_batch).detach().cpu())
    return torch.cat(mapped, dim=0)


def neural_pullback(y, batch_size=plot_batch_size):
    mapped = []
    for start in range(0, y.shape[0], batch_size):
        y_batch = y[start:start + batch_size]
        x_batch = dataset.sample_x(y_batch.shape[0])
        mapped.append(predictor.pullback(x=x_batch, y=y_batch).detach().cpu())
    return torch.cat(mapped, dim=0)


def circle_boundary(center, radius, n_points):
    theta = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=True)
    boundary = np.column_stack([np.cos(theta), np.sin(theta)])
    boundary = np.asarray(center, dtype=float) + radius * boundary
    return torch.as_tensor(boundary, device=dataset.device, dtype=dataset.dtype)


modes = dataset.modes.detach().cpu().numpy()
r_source = float(np.sqrt(chi2.ppf(coverage_mass, df=dataset.y_dim)))
r_hdr, hdr_radius_mc_mass = calibrate_hdr_radius(coverage_mass, n_radius_mc)

source_ball_boundary = circle_boundary(
    center=(0.0, 0.0),
    radius=r_source,
    n_points=n_boundary_points,
)
pushforward_ball_contour = neural_pushforward(source_ball_boundary).numpy()

pullback_hdr_contours = []
for mode in modes:
    target_hdr_boundary = circle_boundary(
        center=mode,
        radius=r_hdr,
        n_points=n_boundary_points,
    )
    pullback_hdr_contours.append(neural_pullback(target_hdr_boundary).numpy())

print(f"coverage_mass = {coverage_mass:.3f}")
print(f"r_source from chi2 quantile = {r_source:.4f}")
print(f"r_hdr from MC calibration = {r_hdr:.4f}")
print(f"MC mass at r_hdr = {hdr_radius_mc_mass:.4f}")

fig, (ax_source, ax_target) = plt.subplots(
    1,
    2,
    figsize=(13.0, 6.0),
    constrained_layout=True,
)

ax_source.add_patch(
    Circle(
        (0.0, 0.0),
        r_source,
        fill=False,
        edgecolor="forestgreen",
        linewidth=2.6,
    )
)
for contour in pullback_hdr_contours:
    ax_source.plot(
        contour[:, 0],
        contour[:, 1],
        color="firebrick",
        linewidth=2.2,
    )
ax_source.set_title("Source Gaussian")
ax_source.set_xlabel(r"$u_1$")
ax_source.set_ylabel(r"$u_2$")
ax_source.set_xlim(source_bounds[0], source_bounds[1])
ax_source.set_ylim(source_bounds[2], source_bounds[3])
ax_source.set_aspect("equal", adjustable="box")
ax_source.legend(
    handles=[
        Line2D([0], [0], color="forestgreen", lw=2.6,
               label=r"$B(0, r_{1-\alpha})$"),
        Line2D([0], [0], color="firebrick", lw=2.2,
               label=r"$Q_\theta^{-1}($HDR$_{1-\alpha})$"),
    ],
    loc="upper right",
    frameon=False,
)

ax_target.plot(
    pushforward_ball_contour[:, 0],
    pushforward_ball_contour[:, 1],
    color="forestgreen",
    linewidth=2.8,
)
for mode in modes:
    ax_target.add_patch(
        Circle(
            tuple(mode),
            r_hdr,
            fill=False,
            edgecolor="firebrick",
            linewidth=2.4,
        )
    )
ax_target.set_title("Target bimodal Gaussian")
ax_target.set_xlabel(r"$y_1$")
ax_target.set_ylabel(r"$y_2$")
ax_target.set_xlim(target_bounds[0], target_bounds[1])
ax_target.set_ylim(target_bounds[2], target_bounds[3])
ax_target.set_aspect("equal", adjustable="box")
ax_target.legend(
    handles=[
        Line2D([0], [0], color="forestgreen", lw=2.8,
               label=r"$Q_\theta(B(0, r_{1-\alpha}))$"),
        Line2D([0], [0], color="firebrick", lw=2.4,
               label=r"HDR$_{1-\alpha}$"),
    ],
    loc="upper right",
    frameon=False,
)

fig.suptitle(
    r"Neural OT map $Q_\theta$, $\alpha$=" + f"{alpha:.2f}",
    y=1.02,
)
plt.show()